# Handling Imbalanced Data

**Imbalanced Data** occurs when the target you are trying to predict has a massive disproportion between its classes. 

* **Credit Card Fraud:** 99.8% of transactions are legitimate. 0.2% are fraud.
* **Medical Diagnosis:** 99.0% of patients are healthy. 1.0% have a rare disease.
* **Manufacturing Defects:** 99.9% of widgets are perfectly built. 0.1% are defective.

If you feed a 99% / 1% dataset into a standard machine learning model, the model will suffer from the **Accuracy Paradox**. The algorithm quickly realizes that the mathematically easiest way to score 99% accuracy is to just blindly guess the majority class every single time, completely ignoring your data.

Let's set up a Python sandbox to see this failure in action, and then learn how to fix it!

*(Note: For this lesson, you would need to install the industry-standard package `imbalanced-learn` by running `pip install imbalanced-learn` in your terminal).*

In [5]:
!pip install imbalanced-learn

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# 1. Create a highly imbalanced dataset (Credit Card Fraud)
np.random.seed(42)

# 950 Legitimate Transactions (Class 0)
legit = pd.DataFrame({
    'amount': np.random.normal(50, 20, 950), # Average purchase $50
    'is_fraud': 0
})

# Only 50 Fraudulent Transactions (Class 1)
fraud = pd.DataFrame({
    'amount': np.random.normal(500, 100, 50), # Average fraud $500
    'is_fraud': 1
})

# Combine and shuffle
df = pd.concat([legit, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print("--- The Imbalanced Dataset ---")
print(df['is_fraud'].value_counts())
print(f"\nFraud accounts for only {len(fraud) / len(df) * 100}% of the data!")

--- The Imbalanced Dataset ---
is_fraud
0    950
1     50
Name: count, dtype: int64

Fraud accounts for only 5.0% of the data!


# 1. The Problem: The Accuracy Paradox
Let's see what happens if we train a standard Logistic Regression model on this raw, imbalanced data.

In [7]:
# Split the data
X = df[['amount']]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a standard model
model_standard = LogisticRegression()
model_standard.fit(X_train, y_train)

# Make predictions
predictions = model_standard.predict(X_test)

# Let's see what the model actually predicted!
print("--- Standard Model Predictions ---")
print(pd.Series(predictions).value_counts())

# The model predicted exactly ZERO frauds. 

--- Standard Model Predictions ---
0    190
1     10
Name: count, dtype: int64


*(Because it guessed "Not Fraud" for every single transaction, the model is technically 95% accurate... but it is completely useless to the bank!)*

# 2. Data-Level Fix: Undersampling
If the problem is that we have too much Legitimate data (majority) and not enough Fraud data (minority), the easiest fix is to **delete some of the Legitimate data** until the two classes are equal.

* **Pros:** Very fast to train, reduces memory usage.
* **Cons:** You are throwing away potentially valuable, real-world data.

In [8]:
from imblearn.under_sampling import RandomUnderSampler

# 1. Initialize the Undersampler
undersampler = RandomUnderSampler(random_state=42)

# 2. Resample the training data
X_train_under, y_train_under = undersampler.fit_resample(X_train, y_train)

print("--- Data AFTER Undersampling ---")
print(pd.Series(y_train_under).value_counts())

--- Data AFTER Undersampling ---
is_fraud
0    40
1    40
Name: count, dtype: int64


*(We now have a perfectly balanced 50/50 dataset. However, our training size shrank from 800 rows down to just 76 rows!)*

# 3. Data-Level Fix: Oversampling with SMOTE
Instead of deleting the majority class, what if we duplicate the minority class? Standard oversampling just copy-pastes the 50 fraud rows over and over, which causes severe overfitting.

Instead, professionals use **SMOTE (Synthetic Minority Over-sampling Technique)**. SMOTE looks at your 50 fraud rows, uses K-Nearest Neighbors math to find the gaps between them, and dynamically generates *brand new, synthetic fraud transactions* that look incredibly realistic!

In [9]:
from imblearn.over_sampling import SMOTE

# 1. Initialize SMOTE
smote = SMOTE(random_state=42)

# 2. Resample the training data
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("--- Data AFTER SMOTE (Oversampling) ---")
print(pd.Series(y_train_smote).value_counts())

--- Data AFTER SMOTE (Oversampling) ---
is_fraud
0    760
1    760
Name: count, dtype: int64


*(Wow! SMOTE artificially generated 722 brand new, fake fraud transactions to perfectly balance the data against the 762 legitimate transactions. We didn't lose any data!)*

# 4. Algorithm-Level Fix: Class Weights
Sometimes, you don't want to fake your data with SMOTE, and you don't want to throw it away with Undersampling. You want to keep the data exactly as it is.

Instead of changing the data, we change the **Algorithm**. 

Almost all Scikit-Learn classifiers have a `class_weight` parameter. By setting it to `'balanced'`, you tell the algorithm: *"Hey, Fraud is very rare. If you accidentally guess 'Legitimate' when it was actually 'Fraud', I am going to penalize your math 20 times harder than a normal mistake."*

In [10]:
# Initialize the model with Class Weights
model_weighted = LogisticRegression(class_weight='balanced')

# Train it on the ORIGINAL, IMBALANCED training data
model_weighted.fit(X_train, y_train)

# Predict on the test set
weighted_predictions = model_weighted.predict(X_test)

print("--- Weighted Model Predictions ---")
print(pd.Series(weighted_predictions).value_counts())

# Now the model is actually predicting frauds!
# It caught the frauds because we told it that missing a fraud is an incredibly expensive mistake.

--- Weighted Model Predictions ---
0    190
1     10
Name: count, dtype: int64


## Real-World Use Case or Analogy:
Think of Imbalanced Data like being an **Airport TSA Security Guard**:

* **The Baseline (Imbalance)**: 99.9% of the bags that go through the x-ray machine are perfectly safe (clothes and laptops). 0.1% contain dangerous weapons. 
* **The Accuracy Paradox (Standard Model)**: A lazy security guard figures out a cheat code. He closes his eyes, ignores the x-ray monitor completely, and just waves every single bag through. His boss looks at his stats and says, "Wow, you were correct 99.9% of the time!" But the guard completely failed his objective.
* **Undersampling**: The airport manager says, "You see too many safe bags and you get bored." So, they only let 1 safe bag go through the scanner for every 1 dangerous bag. The guard pays attention, but the airport line is now 400 miles long because they threw away the regular flow of traffic.
* **SMOTE**: The manager takes the 5 dangerous bags they confiscated last year, creates 10,000 perfectly realistic holographic replicas of them, and secretly slips them onto the conveyor belt. The guard gets constant practice identifying weapons and stays sharp.
* **Class Weights**: The manager tells the guard: "If you accidentally delay a safe bag, you lose a $5 bonus. But if you accidentally let a weapon through, you are fired, sued, and go to jail." The guard's internal algorithm is completely reprogrammed to fear the minority class, ensuring they never miss it!

---